# Exploratory Data Analysis Section

This notebook will use PyAthena to query a AWS Glue database table that contains the raw information then perform Exploratory Data Analysis.

In [1]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import boto3

from windrose import WindroseAxes

In [ ]:
# read in the csv data file into a pandas dataframe
raw_df = pd.read_csv("../2_raw_data/raw_merged/raw_merged_data.csv")

In [ ]:
# Get shape of the data
raw_df.shape

In [ ]:
# Get a list of the column names
raw_df.columns

In [ ]:
# get information on the df
raw_df.info()

In [ ]:
# get number of nulls in each column
raw_df.isnull().sum()

In [ ]:
# get summary statistics of the dataframe
raw_df.describe()

## Target Variable Exploration of Delayed and Cancelled flights

In [ ]:
# count the number of DEP_DEL15 values
raw_df['DEP_DEL15'].value_counts()

In [ ]:
# get unique values of DEP_DEL15
raw_df['DEP_DEL15'].unique()

In [ ]:
# count the number of CANCELLED values
raw_df['CANCELLED'].value_counts()

In [ ]:
# visualize the distribution of delayed flights
plt.figure(figsize=(10,6))
sns.countplot(data=raw_df, x='DEP_DEL15')
plt.title('Distribution of Delayed Flights')
plt.xlabel('Delayed (1 = Yes, 0 = No)')
plt.ylabel('Number of Flights')
plt.tight_layout()
plt.show()

In [ ]:
# visualize distribution of canceled flights
plt.figure(figsize=(10,6))
sns.countplot(data=raw_df, x='CANCELLED')
plt.title('Distribution of Canceled Flights')
plt.xlabel('Canceled (1 = Yes, 0 = No)')
plt.ylabel('Number of Flights')
plt.tight_layout()
plt.show()

In [ ]:
# explore the distribution of carriers and flights
# get count of flights by airline
carrier_counts = raw_df['OP_UNIQUE_CARRIER'].value_counts().reset_index()
carrier_counts.columns = ['Carrier', 'Count of Flights']

# plot the count of flights by airline
plt.figure(figsize=(10,6))
sns.barplot(data=carrier_counts, x='Carrier', y='Count of Flights')
plt.title('Count of Flights by Carrier')
plt.xlabel('Carrier')
plt.ylabel('Count of Flights')
plt.tight_layout()
plt.show()

In [ ]:
# create new category in dataframe for eda
raw_eda_df = raw_df.copy()

In [ ]:
"""
Funciton that will categorize Canceled, Delayed, and On-Time flights
"""
def classify_fl_status(row):
    if row['CANCELLED'] == 1:
        return 'Cancelled'
    elif row['DEP_DEL15'] == 1:
        return 'Delayed'
    elif row['DEP_DEL15'] == 0:
        return 'On-Time'
    else:
        return 'Unknown'

In [ ]:
# create the new column in the dataframe by applying the function
raw_eda_df['Flight_Status'] = raw_eda_df.apply(classify_fl_status, axis=1)

# get the counts of the flight status column
fl_status_counts = raw_eda_df['Flight_Status'].value_counts().reset_index()
# print(fl_status_counts)

# print the percentage of each flight status
fl_status_counts['Percentage'] = (fl_status_counts['count'] / fl_status_counts['count'].sum()) * 100
print(fl_status_counts)

# visualize the distribution of flight status
plt.figure(figsize=(10,6))
sns.countplot(data=raw_eda_df, x='Flight_Status')
plt.title('Distribution of Flight Status')
plt.xlabel('Flight Status')
plt.ylabel('Number of Flights')
plt.tight_layout()
plt.show()

In [ ]:
# explore flight carriers and statuses
# get the count of flights by carrier and status
carrier_status_counts = raw_eda_df.groupby(['OP_UNIQUE_CARRIER', 'Flight_Status']).size().reset_index()
carrier_status_counts.columns = ['Carrier', 'Flight_Status', 'Count']

# pivot table for plotting
carrier_status_pivot = carrier_status_counts.pivot(index='Carrier', columns='Flight_Status', values='Count').fillna(0)
print(carrier_status_pivot)
# sort by total flight count per carrier
carrier_status_pivot['Total'] = carrier_status_pivot.sum(axis=1)
carrier_status_pivot = carrier_status_pivot.sort_values(by='Total', ascending=False)
carrier_status_pivot = carrier_status_pivot.drop(columns='Total')

# plot stacked bar chart
carrier_status_pivot.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Count of Flights by Carrier and Flight Status')
plt.xlabel('Carrier')
plt.ylabel('Number of Flights')
plt.legend(title='Flight Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# noticing mismatch in counts before and after 
# count how many rows have both cancelled and delayed
both_cancelled_and_delayed = raw_eda_df[
    (raw_eda_df['DEP_DEL15'] == 1) & 
    (raw_eda_df['CANCELLED'] == 1)
]

print("Rows where a cancelled flight was also marked as delayed:", len(both_cancelled_and_delayed))

## There are 10 flights that were classified as both cancelled and delayed,
## this accounts for why the counts seem to be off when we examined the counts of the columns previously
## these rows will be classified as cancelled in the processed dataset

In [ ]:
# one hot encode the carrier column
carrier_dummies = pd.get_dummies(raw_eda_df['OP_UNIQUE_CARRIER'], prefix='Carrier')

# concatenate the carrier dummies with the original dataframe, we are not going to get rid of the extra column just yet.
raw_eda_df = pd.concat([raw_eda_df, carrier_dummies], axis=1)


print(raw_eda_df.head())

### Exploration of Weather Features

In [ ]:
# get dtype of weather columns
weather_columns = ['wind_dir_degrees', 'wind_speed_kt', 'visibility_statute_mi', 'temperature_c', 'dewpoint_c', 'altimeter_hpa']
raw_eda_df[weather_columns].dtypes

In [ ]:
# check unique values for wind direction
raw_eda_df['wind_dir_degrees'].unique()     

In [ ]:
# since 0 and 360 are the same, we will replace 0 with 360
raw_eda_df['wind_dir_degrees'] = raw_eda_df['wind_dir_degrees'].replace(0.0, 360.0)

# Address missing values in wind_dir_degrees to change to integer and address missing values
# Missing values are attributed to what is known as Variable Wind speeds, which are often associated with relatively calm wind speeds which are less than 6 knots
# We will replace these missing values with the mode of the wind_dir_degrees column
# get the mode of wind_dir_degrees
mode_wind_dir = raw_eda_df['wind_dir_degrees'].mode()[0]
print("Mode of wind_dir_degrees:", mode_wind_dir)

# replace missing values with the mode
raw_eda_df['wind_dir_degrees'] = raw_eda_df['wind_dir_degrees'].fillna(mode_wind_dir)

# get the count of missing values in wind_dir_degrees
missing_wind_dir = raw_eda_df['wind_dir_degrees'].isnull().sum()
print("Missing values in wind_dir_degrees:", missing_wind_dir)

In [ ]:
# convert the wind_dir_degrees column to integer
raw_eda_df['wind_dir_degrees'] = raw_eda_df['wind_dir_degrees'].astype(int)

In [ ]:
# explore unique values for wind speed
print(raw_eda_df['wind_speed_kt'].unique())

# convert them to integers
raw_eda_df['wind_speed_kt'] = raw_eda_df['wind_speed_kt'].astype(int)

In [ ]:
# explore unique values for wind gust
print(raw_eda_df['wind_gust_kt'].unique())

# convert them to integers
raw_eda_df['wind_gust_kt'] = raw_eda_df['wind_gust_kt'].astype(int)

In [ ]:
# explore unique values for visibility
print(raw_eda_df['visibility_statute_mi'].unique())

In [ ]:
# check temperature unique values
print(raw_eda_df['temperature_c'].unique())

In [ ]:
# check dewpoint unique values
print(raw_eda_df['dewpoint_c'].unique())

In [ ]:
# check altimeter unique values
print(raw_eda_df['altimeter_hpa'].unique())

In [ ]:
# develop a windrose visualization to visualize the most common direction and speeds.
from windrose import WindroseAxes

wind_rose_df = raw_eda_df[['wind_dir_degrees', 'wind_speed_kt']]

ax = WindroseAxes.from_ax()
ax.bar(
    wind_rose_df['wind_dir_degrees'],
    wind_rose_df['wind_speed_kt'],
    normed=True,
    opening=0.8,
    edgecolor='white'
)
ax.set_title('Wind Rose – San Diego Airport')
ax.set_legend(title="Wind Speed (kt)")
plt.show()

## Destination Exploration

In [ ]:
# check the unique values for DEST
print(raw_eda_df['DEST'].unique())

In [ ]:
# explore visualization of flights by destination
plt.figure(figsize=(12, 6))
sns.countplot(data=raw_eda_df, x='DEST', order=raw_eda_df['DEST'].value_counts().index)
plt.title('Count of Flights by Destination')
plt.xlabel('Destination Airport')
plt.ylabel('Number of Flights')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# map airport codes to region
region_map = {
    # West
    'LAX': 'West', 'SFO': 'West', 'OAK': 'West', 'SJC': 'West', 'SMF': 'West',
    'SBP': 'West', 'STS': 'West', 'SAN': 'West', 'FAT': 'West', 'MRY': 'West',
    'BOI': 'West', 'PDX': 'West', 'RDM': 'West', 'EUG': 'West', 'BLI': 'West',
    'PAE': 'West', 'RNO': 'West', 'PSC': 'West', 'SMF': 'West', 'SEA': 'West',
    'GEG': 'West', 'MFR': 'West', 

    # Mountain
    'SLC': 'Mountain', 'DEN': 'Mountain', 'COS': 'Mountain', 'HDN': 'Mountain',
    'ABQ': 'Mountain', 'BZN': 'Mountain', 'JAC': 'Mountain', 'IDA': 'Mountain',
    'MSO': 'Mountain', 'FCA': 'Mountain', 'PVU': 'Mountain', 'EGE': 'Mountain',

    # Southwest
    'PHX': 'Southwest', 'TUS': 'Southwest', 'LAS': 'Southwest', 'AZA': 'Southwest',
    'ELP': 'Southwest',

    # Midwest
    'ORD': 'Midwest', 'MDW': 'Midwest', 'MCI': 'Midwest', 'STL': 'Midwest',
    'DTW': 'Midwest', 'MSP': 'Midwest', 'IND': 'Midwest', 'CMH': 'Midwest',
    'CLE': 'Midwest', 'MKE': 'Midwest', 'DSM': 'Midwest', 'FSD': 'Midwest',

    # South
    'CLT': 'South', 'ATL': 'South', 'BNA': 'South', 'DFW': 'South', 'DAL': 'South', 'AUS': 'South',
    'IAH': 'South', 'HOU': 'South', 'SAT': 'South', 'MSY': 'South', 'MCO': 'South',
    'TPA': 'South', 'FLL': 'South', 'MIA': 'South',

    # Northeast
    'BOS': 'Northeast', 'PHL': 'Northeast', 'JFK': 'Northeast', 'EWR': 'Northeast',
    'PIT': 'Northeast', 'IAD': 'Northeast', 'BWI': 'Northeast',

    # OCONUS will serve as the placeholder for Alaska and Hawaii flights
    # OCONUS refers to Outside Continental United States
    'ANC': 'OCONUS', 'HNL': 'OCONUS', 'KOA': 'OCONUS', 'OGG': 'OCONUS', 'LIH': 'OCONUS',
}
# DEST to region
raw_eda_df['DEST_REGION'] = raw_eda_df['DEST'].map(region_map).fillna('Other')

# results
print(raw_eda_df[['DEST', 'DEST_REGION']].head())

In [ ]:
# visualize the distribution of flights by region
plt.figure(figsize=(10,6))
sns.countplot(data=raw_eda_df, x='DEST_REGION', order=raw_eda_df['DEST_REGION'].value_counts().index)
plt.title('Count of Flights by Destination Region')
plt.xlabel('Destination Region')
plt.ylabel('Number of Flights')
plt.tight_layout()
plt.show()

# print counts of flights by region
print(raw_eda_df['DEST_REGION'].value_counts())

In [ ]:
# creating a stacked bar chart of destination region and flight status
region_status_counts = raw_eda_df.groupby(['DEST_REGION', 'Flight_Status']).size().reset_index()
region_status_counts.columns = ['Region', 'Flight_Status', 'Count']

# pivot table for plotting
region_status_pivot = region_status_counts.pivot(index='Region', columns='Flight_Status', values='Count').fillna(0)
print(region_status_pivot)

# sort by total flight count per region
region_status_pivot['Total'] = region_status_pivot.sum(axis=1)
region_status_pivot = region_status_pivot.sort_values(by='Total', ascending=False)
region_status_pivot = region_status_pivot.drop(columns='Total')

# plot stacked bar chart
region_status_pivot.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Count of Flights by Destination Region and Flight Status')
plt.xlabel('Destination Region')
plt.ylabel('Number of Flights')
plt.legend(title='Flight Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Date Exploration

In [33]:
# One hot encode the DEST_REGION column
dest_region_dummies = pd.get_dummies(raw_eda_df['DEST_REGION'], prefix='Dest_Region')

# concatenate the carrier dummies with the original dataframe, we are not going to get rid of the extra column just yet.
raw_eda_df = pd.concat([raw_eda_df, dest_region_dummies], axis=1)

In [ ]:
print(raw_eda_df.dtypes)

## Finalize data frame for processed data

In [34]:
processed_df = raw_eda_df.copy()
processed_df = processed_df.drop(columns=['DEP_TIME', 'DEP_DEL15', 'CANCELLED', 'OP_CARRIER_FL_NUM', 'CANCELLATION_CODE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'])

# create the target variable for the model using Flight_Status where if Cancelled or Delayed = 1 and On-time = 0
processed_df['Flight_Status_Binary'] = processed_df['Flight_Status'].apply(
    lambda x: 1 if x in ['Delayed', 'Cancelled'] else 0
)

#print head
print(processed_df.head())

      FL_DATE CRS_DEP_TIME OP_UNIQUE_CARRIER DEST  wind_dir_degrees  \
0  2023-01-01     06:10:00                DL  DTW               290   
1  2023-01-01     06:15:00                DL  SLC               290   
2  2023-01-01     06:15:00                AA  CLT               290   
3  2023-01-01     06:15:00                DL  ATL               290   
4  2023-01-01     06:15:00                DL  MSP               290   

   wind_speed_kt  wind_gust_kt  visibility_statute_mi  temperature_c  \
0              7            19                    6.0           13.3   
1              7            19                    6.0           13.3   
2              7            19                    6.0           13.3   
3              7            19                    6.0           13.3   
4              7            19                    6.0           13.3   

   dewpoint_c  ...  Carrier_WN DEST_REGION  Dest_Region_Midwest  \
0        10.0  ...       False     Midwest                 True   
1     

In [35]:
# save the processed data to a csv file
processed_df.to_csv("processed_data/processed_data.csv", index=False)

## Upload processed_data.csv to Processed_data S3 Bucket

In [2]:
# examine list of buckets available
s3 = boto3.client('s3')
response = s3.list_buckets()
print("Existing buckets:")
for bucket in response['Buckets']:
    print(bucket['Name'])

Existing buckets:
group9-ml-proj-athena-query-bucket-grw
group9-ml-proj-model-artifacts-bucket-grw
group9-ml-proj-monitoring-bucket-grw
group9-ml-proj-processed-data-bucket-grw
group9-ml-proj-raw-data-bucket-grw
group9-ml-proj-training-artifacts-bucket-grw
sagemaker-studio-7sb8gxpljq9
sagemaker-studio-chxawrkbs1q
sagemaker-studio-o4i4ukj5y3b
sagemaker-us-east-1-804823187130


In [3]:
# set local path of organized weather CSV file
local_file_path = 'processed_data/processed_data.csv'

In [4]:
# set the raw data bucket variable
processed_bucket = "group9-ml-proj-processed-data-bucket-grw"

In [5]:
# set the S3 key path, there will be 
s3_key = "all_data/processed_data.csv"

In [6]:
# upload file to the S3 bucket
try:
    s3.upload_file(local_file_path, processed_bucket, s3_key)
    print(f"Uploaded {local_file_path} to s3://{processed_bucket}/{s3_key}")
except Exception as e:
    print("Error uploading file to S3:", e)

Uploaded processed_data/processed_data.csv to s3://group9-ml-proj-processed-data-bucket-grw/all_data/processed_data.csv


## skip this section and only work above on the EDA (this is the failed trying to read in from the database)


#set parameters for athena connection
region_name = "us-east-1"
s3_staging_dir = "s3://group9-ml-proj-raw-data-bucket-grw/queries/"


#establish connection
conn = connect(s3_staging_dir=s3_staging_dir, region_name=region_name)
cursor = conn.cursor()

#query the glue table
query = """
    SELECT * 
    FROM raw_data.raw_merged_data_csv
    """


#read the query into a pandas dataframe
raw_df = pd.read_sql(query, conn)

#print head of the df
raw_df.head()

In [1]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>